In [22]:
import numpy as np
import pandas as pd

from nilearn import datasets
from nilearn.maskers import NiftiMasker
from nilearn.image import index_img, resample_to_img, math_img

from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings("ignore")

In [23]:
haxby = datasets.fetch_haxby(subjects=[1,2,3,4,5,6], verbose=0)

print("Subjects:", len(haxby.func))

Subjects: 6


In [24]:
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear"))
])

In [ ]:
def compute_subject_accuracy(func_img, mask_img, labels, runs):
    
    masker = NiftiMasker(mask_img=mask_img, standardize=True)
    X = masker.fit_transform(func_img)
    
    y = labels
    groups = runs
    
    logo = LeaveOneGroupOut()
    
    accuracies = []
    
    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
    
    return np.mean(accuracies)

In [26]:
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm', verbose=0)
atlas_img = atlas.maps
labels = atlas.labels

v1_index = labels.index('Occipital Pole')
v1_mask_atlas = math_img("img == %d" % v1_index, img=atlas_img)

In [ ]:
vtc_accuracies = []
v1_accuracies = []

for i in range(len(haxby.func)):
    
    func_img = haxby.func[i]
    
    behavior = pd.read_csv(haxby.session_target[i], sep=" ")
    
    labels = behavior["labels"].values
    runs = behavior["chunks"].values
    
    non_rest = labels != "rest"
    labels = labels[non_rest]
    runs = runs[non_rest]
    
    func_img = index_img(func_img, np.where(non_rest)[0])
    

    vtc_mask = haxby.mask_vt[i]
    v1_mask_resampled = resample_to_img(
        v1_mask_atlas,
        func_img,
        interpolation="nearest"
    )
    
    acc_vtc = compute_subject_accuracy(func_img, vtc_mask, labels, runs)
    acc_v1 = compute_subject_accuracy(func_img, v1_mask_resampled, labels, runs)
    
    vtc_accuracies.append(acc_vtc)
    v1_accuracies.append(acc_v1)

In [28]:
vtc_accuracies = np.array(vtc_accuracies)
v1_accuracies = np.array(v1_accuracies)

print("VTC accuracies:", vtc_accuracies)
print("V1 accuracies:", v1_accuracies)

print("\nMean VTC accuracy:", vtc_accuracies.mean())
print("Mean V1 accuracy:", v1_accuracies.mean())

VTC accuracies: [0.68171296 0.57175926 0.63773148 0.46412037 0.69191919 0.73148148]
V1 accuracies: [0.13657407 0.17361111 0.12615741 0.15509259 0.11616162 0.15625   ]

Mean VTC accuracy: 0.6297874579124579
Mean V1 accuracy: 0.14397446689113358


In [29]:
t_stat, p_value = ttest_rel(vtc_accuracies, v1_accuracies)

print("Paired t-test")
print("t =", t_stat)
print("p =", p_value)

Paired t-test
t = 10.935612826489347
p = 0.00011115904718152355
